# Weighted Log-Odds Analysis — WildChat Demographics Analysis

This notebook computes weighted log-odds ratios to compare topic usage between:
- High vs. low **education** states (top/bottom quartile by % bachelor's degree or higher)
- High vs. low **income** states (top/bottom quartile by median household income)

**Reference:** Monroe et al. (2008). Fightin' Words: Lexical Feature Selection and Evaluation for Identifying the Content of Political Conflict.

**Inputs:**
- `results/state_topic_proportions.parquet` — from BERTopic pipeline
- `data/acs_covariates.csv` — ACS education + income data (Person C's data)

**Outputs:**
- `results/log_odds_education.csv` — log-odds by education group
- `results/log_odds_income.csv` — log-odds by income group
- `results/log_odds_figures/` — bar charts for paper

## 0. Install dependencies

In [ ]:
!pip install pandas numpy matplotlib seaborn scipy pyarrow

## 1. Imports & setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

Path("results/log_odds_figures").mkdir(parents=True, exist_ok=True)

print("All imports OK")

## 2. Load data

In [ ]:
# BERTopic state-level topic proportions
topic_props = pd.read_parquet("results/state_topic_proportions.parquet")

print(f"Topic proportions shape: {topic_props.shape}")
print(f"States: {topic_props['state'].nunique()}")
print(f"\nFirst few rows:")
print(topic_props.head())

In [ ]:
# ACS demographic covariates 
# Columns: state, income, education
acs = pd.read_csv("../../data/state_covariates.csv")

print(f"ACS data shape: {acs.shape}")
print(f"Columns: {acs.columns.tolist()}")
print(acs.head())

In [ ]:
# Merge topic proportions with ACS covariates
df = topic_props.merge(acs, on='state', how='inner')

print(f"Merged shape: {df.shape}")
print(f"States after merge: {df['state'].nunique()}")

# Get topic columns
topic_cols = [c for c in df.columns if c.startswith('topic_')]
print(f"Topic columns: {len(topic_cols)}")

## 3. Define quartile groups

We split states into top and bottom quartiles for each demographic variable.
Middle two quartiles are excluded to make the contrast sharper and more interpretable.

In [ ]:
def get_quartile_groups(df, col):
    """
    Split states into top quartile (Q4) and bottom quartile (Q1).
    Returns two dataframes: high_group, low_group
    """
    q25 = df[col].quantile(0.25)
    q75 = df[col].quantile(0.75)

    high = df[df[col] >= q75].copy()
    low  = df[df[col] <= q25].copy()

    print(f"\n{col}:")
    print(f"  Q75 threshold : {q75:.2f}")
    print(f"  Q25 threshold : {q25:.2f}")
    print(f"  High group    : {len(high)} states — {sorted(high['state'].tolist())}")
    print(f"  Low group     : {len(low)} states — {sorted(low['state'].tolist())}")

    return high, low


# Education groups
edu_high, edu_low = get_quartile_groups(df, 'education')

# Income groups
inc_high, inc_low = get_quartile_groups(df, 'income')

## 4. Weighted log-odds function

We use the weighted log-odds ratio with an uninformative Dirichlet prior (Monroe et al., 2008).
This is more robust than raw log-odds because it accounts for variance in low-frequency topics.

For each topic t:
- y1 = total topic t usage in group 1 (high)
- y2 = total topic t usage in group 2 (low)
- n1, n2 = total usage across all topics in each group
- z-score > 1.96 indicates significantly higher usage in group 1

In [ ]:
def weighted_log_odds(group1_props, group2_props, topic_cols, alpha=0.01):
    """
    Compute weighted log-odds z-scores for each topic.
    Positive z = topic more common in group1 (high).
    Negative z = topic more common in group2 (low).

    Parameters:
        group1_props: dataframe of high group (states × topic proportions)
        group2_props: dataframe of low group (states × topic proportions)
        topic_cols: list of topic column names
        alpha: Dirichlet prior concentration (default 0.01, uninformative)
    """
    # Sum topic proportions across states in each group
    y1 = group1_props[topic_cols].sum().values
    y2 = group2_props[topic_cols].sum().values

    n1 = y1.sum()
    n2 = y2.sum()
    k  = len(topic_cols)

    # Prior counts
    a0 = alpha * k

    # Log-odds with prior
    log_odds = (
        np.log((y1 + alpha) / (n1 + a0 - y1 - alpha)) -
        np.log((y2 + alpha) / (n2 + a0 - y2 - alpha))
    )

    # Variance estimate
    variance = (
        1.0 / (y1 + alpha) +
        1.0 / (y2 + alpha)
    )

    z_scores = log_odds / np.sqrt(variance)

    results = pd.DataFrame({
        'topic': topic_cols,
        'log_odds': log_odds,
        'z_score': z_scores,
        'group1_mean': group1_props[topic_cols].mean().values,
        'group2_mean': group2_props[topic_cols].mean().values,
    })

    results = results.sort_values('z_score', ascending=False).reset_index(drop=True)
    return results


print("Weighted log-odds function defined.")

## 5. Run log-odds for education

In [ ]:
edu_results = weighted_log_odds(edu_high, edu_low, topic_cols)

# Mark significance (|z| > 1.96 ≈ p < 0.05)
edu_results['significant'] = edu_results['z_score'].abs() > 1.96
edu_results['direction'] = edu_results['z_score'].apply(
    lambda z: 'high_edu' if z > 0 else 'low_edu'
)

edu_results.to_csv("results/log_odds_education.csv", index=False)

print(f"Significant topics (|z| > 1.96): {edu_results['significant'].sum()}")
print("\nTop 10 topics more common in HIGH education states:")
print(edu_results.head(10)[['topic', 'z_score', 'group1_mean', 'group2_mean']].to_string(index=False))
print("\nTop 10 topics more common in LOW education states:")
print(edu_results.tail(10)[['topic', 'z_score', 'group1_mean', 'group2_mean']].to_string(index=False))

## 6. Run log-odds for income

In [ ]:
inc_results = weighted_log_odds(inc_high, inc_low, topic_cols)

inc_results['significant'] = inc_results['z_score'].abs() > 1.96
inc_results['direction'] = inc_results['z_score'].apply(
    lambda z: 'high_income' if z > 0 else 'low_income'
)

inc_results.to_csv("results/log_odds_income.csv", index=False)

print(f"Significant topics (|z| > 1.96): {inc_results['significant'].sum()}")
print("\nTop 10 topics more common in HIGH income states:")
print(inc_results.head(10)[['topic', 'z_score', 'group1_mean', 'group2_mean']].to_string(index=False))
print("\nTop 10 topics more common in LOW income states:")
print(inc_results.tail(10)[['topic', 'z_score', 'group1_mean', 'group2_mean']].to_string(index=False))

## 7. Visualize results

In [ ]:
def plot_log_odds(results, title, filename, top_n=15):
    """
    Bar chart showing top_n topics on each side (high vs low group).
    Significant topics are colored, non-significant are grey.
    """
    # Top N from each end
    top = results.head(top_n).copy()
    bottom = results.tail(top_n).copy()
    plot_df = pd.concat([top, bottom]).drop_duplicates(subset='topic')
    plot_df = plot_df.sort_values('z_score')

    colors = plot_df['z_score'].apply(
        lambda z: '#2166ac' if z > 1.96 else ('#d73027' if z < -1.96 else '#aaaaaa')
    )

    fig, ax = plt.subplots(figsize=(10, max(8, len(plot_df) * 0.35)))
    bars = ax.barh(plot_df['topic'], plot_df['z_score'], color=colors)
    ax.axvline(x=1.96,  color='black', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.axvline(x=-1.96, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.axvline(x=0, color='black', linewidth=0.8)

    ax.set_xlabel('Weighted Log-Odds Z-Score', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#2166ac', label='Significantly higher (high group)'),
        Patch(facecolor='#d73027', label='Significantly higher (low group)'),
        Patch(facecolor='#aaaaaa', label='Not significant'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

    plt.tight_layout()
    plt.savefig(f"results/log_odds_figures/{filename}", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved → results/log_odds_figures/{filename}")


plot_log_odds(
    edu_results,
    title="Topic Usage: High vs. Low Education States",
    filename="log_odds_education.png"
)

plot_log_odds(
    inc_results,
    title="Topic Usage: High vs. Low Income States",
    filename="log_odds_income.png"
)

## 8. Permutation test for robustness

For the most significant topics, we run a permutation test to verify the findings hold
beyond the z-score threshold. This randomly shuffles state group labels 1000 times
and checks how often we'd see a z-score this extreme by chance.

In [ ]:
def permutation_test(df, topic_col, group_col, group_high_val, group_low_val,
                     n_permutations=1000, random_state=42):
    """
    Permutation test for a single topic.
    Returns empirical p-value.
    """
    rng = np.random.default_rng(random_state)

    high_vals = df[df[group_col] == group_high_val][topic_col].values
    low_vals  = df[df[group_col] == group_low_val][topic_col].values

    observed_diff = high_vals.mean() - low_vals.mean()

    combined = np.concatenate([high_vals, low_vals])
    n_high = len(high_vals)

    null_diffs = []
    for _ in range(n_permutations):
        shuffled = rng.permutation(combined)
        null_diff = shuffled[:n_high].mean() - shuffled[n_high:].mean()
        null_diffs.append(null_diff)

    null_diffs = np.array(null_diffs)
    p_value = (np.abs(null_diffs) >= np.abs(observed_diff)).mean()

    return observed_diff, p_value


# Add education group label to df for permutation test
df['edu_group'] = None
df.loc[df.index.isin(edu_high.index), 'edu_group'] = 'high'
df.loc[df.index.isin(edu_low.index),  'edu_group'] = 'low'
df_edu = df[df['edu_group'].notna()].copy()

# Run permutation test on top 5 significant education topics
top_edu_topics = edu_results[edu_results['significant']].head(5)['topic'].tolist()

print("Permutation tests for top education topics (1000 permutations each):")
print(f"{'Topic':<30} {'Obs. Diff':>10} {'p-value':>10} {'Sig':>6}")
print("-" * 60)

perm_rows = []
for topic in top_edu_topics:
    obs_diff, p_val = permutation_test(df_edu, topic, 'edu_group', 'high', 'low')
    sig = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else ''))
    print(f"{topic:<30} {obs_diff:>10.4f} {p_val:>10.4f} {sig:>6}")
    perm_rows.append({'topic': topic, 'obs_diff': obs_diff, 'p_value': p_val})

perm_df = pd.DataFrame(perm_rows)
perm_df.to_csv("results/permutation_test_education.csv", index=False)
print("\nSaved → results/permutation_test_education.csv")

## 9. Summary

In [ ]:
print("=" * 55)
print("LOG-ODDS ANALYSIS SUMMARY")
print("=" * 55)
print(f"States analyzed     : {df['state'].nunique()}")
print(f"Topics analyzed     : {len(topic_cols)}")
print(f"")
print(f"Education (high vs low quartile):")
print(f"  High group        : {len(edu_high)} states")
print(f"  Low group         : {len(edu_low)} states")
print(f"  Significant topics: {edu_results['significant'].sum()}")
print(f"")
print(f"Income (high vs low quartile):")
print(f"  High group        : {len(inc_high)} states")
print(f"  Low group         : {len(inc_low)} states")
print(f"  Significant topics: {inc_results['significant'].sum()}")
print(f"")
print("Output files:")
print("  results/log_odds_education.csv")
print("  results/log_odds_income.csv")
print("  results/permutation_test_education.csv")
print("  results/log_odds_figures/log_odds_education.png  <-- for paper")
print("  results/log_odds_figures/log_odds_income.png     <-- for paper")